In [ ]:
#provided preprocessing code:

import pandas as pd

# Load df & check for duplicate rows & null values
df = pd.read_csv("energydata.csv")
print(df.duplicated().sum())
print(df.isna().sum())

# Convert date feature into separate numerical columns
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df = df.drop(['date'], axis=1)

#unique code in the following block

import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = df.drop('Appliances', axis=1)
y = df['Appliances']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Training set:", X_train.shape, " Test set:", X_test.shape)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# K-Fold cross validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Parameter grid for the random forest
param_grid = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 1.0]
    }

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',  # optimise RMSE
    cv=kfold,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)


print("Best parameters:", grid_search.best_params_)
print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("Random Forest Test Performance")
print(f"MAE : {mae:.4f}")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

importances = pd.Series(best_rf.named_steps['rf'].feature_importances_,index=X.columns).sort_values(ascending=False)

print("Top 10 most important features:")
print(importances.head(10))
#run took ~ 13 mins 16 secs

0
date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64
Training set: (13814, 32)  Test set: (5921, 32)
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'rf__max_depth': None, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__n_estimators': 200}
Best CV RMSE: 72.5586
Random Forest Test Performance
MAE : 32.2620
MSE : 4492.7285
RMSE: 67.0278
Top 10 most important features:
hour           0.084659
RH_1           0.044006
T3             0.041096
Press_mm_hg    0.040118
RH_3           0.039905
RH_2        